# Tome 14, Chapter 12: Equidistribution and random polynomials (Python companion)

Exercises first, solutions at the end. We measure discrepancies against the Erdős–Turán inequality, compute the vertical Sato–Tate law for all elliptic curves over a prime field, check that random polynomials mod $p$ factor like random permutations, and estimate how often a random integer quintic has Galois group $S_5$. The computer science application: **generating irreducible polynomials over $\mathbb F_2$**, as needed for CRCs, finite field arithmetic and stream ciphers, with Rabin's test. Requires `numpy` and `sympy`.

In [ ]:
import math, random
from collections import Counter
from math import comb, factorial
import numpy as np
from sympy.polys.galoistools import gf_ddf_zassenhaus, gf_sqf_p
from sympy.polys.domains import ZZ
from sympy import primerange
from sympy.utilities.iterables import partitions

def ftype(coeffs, p):
    # factorization type of a monic polynomial mod p (coefficients highest first), None if not squarefree
    f = [c % p for c in coeffs]
    if not gf_sqf_p(f, p, ZZ): return None
    t = []
    for g, d in gf_ddf_zassenhaus(f, p, ZZ): t += [d] * ((len(g) - 1) // d)
    return tuple(sorted(t))

## Exercises

**P12.1 (★★) Discrepancy.** For $x_n=n\varphi$ ($\varphi$ the golden ratio), $x_n=n\sqrt2$, $x_n=\sqrt n$ and $N$ uniform random points, compute the exact discrepancy $D_N$ over arcs for $N=10^2,10^3,10^4,10^5$, and the bound of Theorem 12.1 with the best $K\le 4000$.

**P12.2 (★★) Vertical Sato–Tate.** For $p=101$ and $p=1009$, compute $a_p(a,b)$ for all $(a,b)\in\mathbb F_p^2$ (with an FFT over $\mathbb F_p$), check Birch's identity $\sum a_p^2=p^2(p-1)$ exactly, and compute the moments of orders $2$, $4$, $6$, $8$ of $a_p/\sqrt p$ over the non-singular curves, against the Catalan numbers $1,2,5,14$ (Figure 12.1).

**P12.3 (★) Polynomials and permutations.** For $n=6$, list the partitions $\lambda$ and compare $\prod_d\binom{N_p(d)}{m_d}/p^n$ with $P(\lambda)$ for $p=7$ and $p=101$ (Theorem 12.4). Then sample $20\,000$ random monic polynomials of degree $6$ over $\mathbb F_{10007}$ and compare the type frequencies with $P(\lambda)$.

**P12.4 (★★) Galois groups of random quintics.** Draw $2000$ monic quintics with coefficients in $[-20,20]$. Using Exercise 12.6, certify $G_f=S_5$ when $f$ is irreducible mod one prime $p<500$, of type $1\,4$ mod another, and of type $1^32$ mod a third (all with $\bar f$ squarefree). What fraction is certified? For the others, which types occur?

**P12.5 (★, CS application) Random irreducible polynomials over $\mathbb F_2$.** Represent polynomials over $\mathbb F_2$ as Python integers (bit $i$ = coefficient of $X^i$). Implement Rabin's test for degree $n=32$: $f$ is irreducible iff $X^{2^{32}}\equiv X\pmod f$ and $\gcd(X^{2^{16}}-X,f)=1$. Draw random monic polynomials of degree $32$ until one is irreducible, $500$ times, and compare the average number of draws with $\frac{2^{32}}{N_2(32)}$.

## Solutions

### P12.1

In [ ]:
def disc(x):
    # exact discrepancy over arcs of points x in [0,1)
    x = np.sort(np.mod(x, 1.0)); N = len(x); i = np.arange(1, N + 1)
    return max((i / N - x).max() - (i / N - x).min() + 1 / N, 0)

def et_bound(x, Kmax=4000):
    N = len(x); ks = np.arange(1, Kmax)
    S = np.array([abs(np.exp(2j * np.pi * k * x).sum()) for k in ks])
    cum = np.cumsum(S / (ks * N)) * 2 / np.pi          # sum_{k<K}
    Ks = np.arange(2, Kmax + 1)
    return (6 / np.sqrt(Ks) + cum[Ks - 2]).min()

rng = np.random.default_rng(12)
phi = (1 + 5 ** 0.5) / 2
for N in [100, 1000, 10000, 100000]:
    n = np.arange(1, N + 1)
    seqs = {"n*phi": n * phi, "n*sqrt2": n * 2 ** 0.5, "sqrt n": np.sqrt(n), "random": rng.random(N)}
    out = []
    for name, x in seqs.items():
        b = et_bound(x) if N <= 10000 else float("nan")
        out.append(f"{name}: D={disc(x):.5f} (ET {b:.3f})")
    print(f"N={N:6d}  " + "  ".join(out))

For $n\varphi$ the discrepancy decreases like $\frac{\log N}N$ (it is about $0.0005$ at $N=10^4$), much faster than random points ($\approx\frac1{\sqrt N}$). The sequence $\sqrt n$ is equidistributed but slowly: its fractional parts sweep $[0,1)$ in runs. Our Erdős–Turán bound is crude: with $K\le4000$ the term $\frac6{\sqrt K}$ alone is $0.095$, so the bound stalls there, while the true discrepancies go much lower. What it shows is the mechanism: small Weyl sums for $k<K$ force a small discrepancy, and the sharp form of Erdős and Turán, with $\frac1K$, would follow the true values much more closely.

### P12.2

In [ ]:
def all_ap(p):
    chi = np.array([0] + [1 if pow(x, (p - 1) // 2, p) == 1 else -1 for x in range(1, p)], dtype=float)
    x = np.arange(p); A = np.empty((p, p), dtype=np.int64)
    fchi = np.fft.fft(chi)
    for a in range(p):
        n = np.bincount((x ** 3 + a * x) % p, minlength=p).astype(float)
        A[a] = -np.round(np.real(np.fft.ifft(np.conj(np.fft.fft(n)) * fchi))).astype(np.int64)
    return A
for p in [101, 1009]:
    A = all_ap(p)
    a, b = np.meshgrid(np.arange(p), np.arange(p), indexing="ij")
    ns = (4 * a ** 3 + 27 * b ** 2) % p != 0
    X = A[ns] / math.sqrt(p)
    print(f"p={p}: sum a_p^2 = {int((A ** 2).sum())}, p^2(p-1) = {p * p * (p - 1)};  curves {ns.sum()};",
          "moments", [round(float(np.mean(X ** k)), 4) for k in (2, 4, 6, 8)], "max |a_p|/2sqrt(p) =", round(float(np.abs(X).max() / 2), 4))

Birch's identity holds exactly, and the moments over non-singular curves are $0.9999$, $1.9997$, $4.9991$, $13.997$ for $p=101$ and $1.0000, 2.0000, 5.0000, 14.00$ for $p=1009$: the Catalan numbers, moments of the semicircle law. Hasse's bound holds for every curve.

### P12.3

In [ ]:
def N_irr(p, d):
    from sympy import mobius, divisors
    return int(sum(mobius(e) * p ** (d // e) for e in divisors(d))) // d

def P_perm(lam):                      # lam: dict d -> m_d
    r = 1.0
    for d, m in lam.items(): r /= d ** m * factorial(m)
    return r

n = 6
print("type          P(lambda)   p=7       p=101")
for lam in partitions(n):
    lam = dict(lam)
    row = [P_perm(lam)]
    for p in (7, 101):
        cnt = 1
        for d, m in lam.items(): cnt *= comb(N_irr(p, d), m)
        row.append(cnt / p ** n)
    name = " ".join(f"{d}^{m}" for d, m in sorted(lam.items()))
    print(f"{name:12s}  {row[0]:.5f}    {row[1]:.5f}   {row[2]:.5f}")

In [ ]:
p = 10007; random.seed(12)
c = Counter(ftype([1] + [random.randrange(p) for _ in range(6)], p) for _ in range(20000))
tot = sum(v for k, v in c.items() if k)
worst = 0
for lam in partitions(6):
    lam = dict(lam); key = tuple(sorted(sum(([d] * m for d, m in lam.items()), [])))
    worst = max(worst, abs(c[key] / tot - P_perm(lam)))
print("non-squarefree:", c[None], " largest |frequency - P(lambda)| over the 11 types:", round(worst, 4))

For $p=101$ the exact proportions agree with $P(\lambda)$ to within $15\%$ in relative terms for the types with many linear factors, such as $1^6$ (choosing $6$ distinct roots among $101$ costs a factor $\prod_{j<6}(1-\frac j{101})\approx0.86$), and to within $1\%$ for the others; for $p=7$ the agreement is rough, as expected from the $O_n(\frac1p)$ error. The sample over $\mathbb F_{10007}$ matches $P(\lambda)$ to within sampling error ($\approx\frac1{\sqrt{20000}}$), and $2$ of the $20\,000$ sampled polynomials had a repeated factor, an event of probability about $\frac1p$.

### P12.4

In [ ]:
random.seed(5)
primes = list(primerange(3, 500))
certified, reducible, other = 0, 0, Counter()
for _ in range(2000):
    f = [1] + [random.randint(-20, 20) for _ in range(5)]
    seen = set()
    for p in primes:
        t = ftype(f, p)
        if t: seen.add(t)
        if {(5,), (1, 4), (1, 1, 1, 2)} <= seen: break
    if {(5,), (1, 4), (1, 1, 1, 2)} <= seen: certified += 1
    elif (5,) not in seen: reducible += 1
    else: other[frozenset(seen)] += 1
print("certified S5:", certified, "  never irreducible mod p (reducible over Q):", reducible, "  irreducible, not certified:", sum(other.values()))
for s, k in other.items(): print(k, sorted(s))

$1883$ of the $2000$ quintics ($94\%$) are certified to have Galois group $S_5$ with primes below $500$. The other $117$ are never irreducible modulo any of these primes, and are reducible over $\Q$ (for instance those with $a_0=0$, about $\frac1{41}$ of them, or with a root $\pm1$). No irreducible quintic with a smaller group turned up, in line with van der Waerden's conjecture: in a box of side $41$ the reducible polynomials, about $\frac C H$ of all, dominate the exceptions.

### P12.5

In [ ]:
def pmulmod(a, b, f, n):
    # product of polynomials a, b over F_2 modulo f (degree n)
    r = 0
    while b:
        if b & 1: r ^= a
        b >>= 1; a <<= 1
        if a >> n & 1: a ^= f
    return r

def pgcd(a, b):
    while b:
        while a and a.bit_length() >= b.bit_length():
            a ^= b << (a.bit_length() - b.bit_length())
        a, b = b, a
    return a

def rabin(f, n=32):
    x = 2; y = x
    for i in range(1, n + 1):
        y = pmulmod(y, y, f, n)                 # y = X^(2^i) mod f
        if i == n // 2 and pgcd(f, y ^ x) != 1: return False
    return y == x

random.seed(32)
draws = []
for _ in range(500):
    k = 0
    while True:
        k += 1
        f = (1 << 32) | random.getrandbits(32)
        if rabin(f): break
    draws.append(k)
print("mean number of draws:", np.mean(draws), "  2^32 / N_2(32) =", 2 ** 32 / N_irr(2, 32))

The average number of draws, about $32$, matches $\frac{2^{32}}{N_2(32)}=32.0000$: a random polynomial of degree $n$ is irreducible with probability almost exactly $\frac1n$, the probability that a random permutation of $S_n$ is an $n$-cycle. Each test costs $n$ modular squarings, so finding an irreducible polynomial costs about $n^2$ squarings: this is why random search is the standard way to build the fields $\mathbb F_{2^n}$ used in coding theory and cryptography, and why the prime number theorem for $\mathbb F_2[X]$ is a complexity statement.